In [0]:
-- It's also possible to create SQL Functions in Databricks.
SELECT * FROM gshen_catalog.enb_projects_workshop.project_master;
SELECT * FROM gshen_catalog.enb_projects_workshop.forecasts;

-- Returns TRUE if the project's EAC exceeds its total budget, otherwise FALSE.
CREATE OR REPLACE FUNCTION gshen_catalog.enb_projects_workshop.budget_status(
    budget_million DOUBLE,
    eac DOUBLE
)
RETURNS STRING
RETURN 
  CASE 
    WHEN eac > budget_million * 1e6 * 1.05 THEN 'Over Budget'
    WHEN eac < budget_million * 1e6 * 0.95 THEN 'Under Budget'
    ELSE 'On Budget'
  END;

-- We can leverage the SQL function in a query
SELECT
  m.`Project ID`,
  m.`Project Name`,
  m.`Total Budget (Million $)`,
  f.`Estimate at Completion ($)`,
  gshen_catalog.enb_projects_workshop.budget_status(m.`Total Budget (Million $)`, f.`Estimate at Completion ($)`) AS budget_risk
FROM gshen_catalog.enb_projects_workshop.project_master m
JOIN gshen_catalog.enb_projects_workshop.forecasts f ON m.`Project ID` = f.`Project ID`;

-- **New** Databricks supports Python UDFs
CREATE OR REPLACE FUNCTION gshen_catalog.enb_projects_workshop.budget_status_pyudf(
    budget_million DOUBLE,
    eac DOUBLE
)
RETURNS STRING
LANGUAGE PYTHON
AS $$
def budget_status(budget_million: float, eac: float) -> str:
    budget = budget_million * 1e6
    if eac > budget * 1.05:
        return 'Over Budget'
    elif eac < budget * 0.95:
        return 'Under Budget'
    else:
        return 'On Budget'
$$

